# 06 · Conectando ao Spark Modo Cluster via Spark Connect 

**Teoria**: docs/05-pyspark-na-pratica.md

**Pré-requisito**: `make spark` (Spark Cluster: 1 master + 2 workers + um servidor Spark Connect, tudo em Docker).

🎯 **Objetivo**: conectar-se a um cluster Spark remoto usando o protocolo gRPC do Spark Connect. 

Seu processo Python aqui é um **cliente gRPC leve** — sem JVM, sem classpath do Hadoop, nada pesado.

Todo o processamento pesado acontece dentro dos containers docker (workers). 
O plano lógico não resolvido via gRPC e recebe os resultados de volta. Isso também significa: 
**os caminhos de arquivos que você referencia devem existir dentro dos 
containers**, não no seu laptop — é exatamente por isso que o `docker-compose.yml` 
faz bind-mount de `./data` em `/data` em cada serviço Spark.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

## Comprovando que a execução realmente acontece no cluster

🎯 **Objetivo**: verificar que o processamento está rodando no cluster Docker, não localmente.


## Lendo o dataset compartilhado

📌 Lembrete: `/data/...` é o caminho **dentro dos containers** (veja o 
mount de volume `x-spark-common` do `docker-compose.yml`).

Execute `make generate-data SCALE=large` no host primeiro — como `./data` está montado 
por bind, os containers o veem imediatamente, sem necessidade de etapa de cópia.


In [ ]:
# Lê os dados Parquet — os arquivos estão no bind-mount /data dentro dos containeres workers
# Caminho ABSOLUTO: o volume `./data:/data` do docker-compose monta na raiz dos containers workers
sdf_vendas = spark.read.parquet("/data/bronze/vendas")

# O count() força a leitura distribuída — cada executor lê uma parte dos arquivos Parquet
print(f"vendas: {sdf_vendas.count():,} rows")
sdf_vendas.show(5)

📌 **Observação**:

O Spark Connect retornou os dados normalmente, como se fosse uma SparkSession local. 
A diferença? Toda a computação (leitura, descompressão, contagem) ocorreu nos workers 
do cluster — seu processo Python apenas recebeu os resultados.

💡 **Dica**: abra a aba **Stages** da Spark UI para ver as Tasks distribuídas entre 
os 2 workers do cluster.

## Tour pelas Partitions

🧠 **Por quê?** — dois níveis diferentes de "RDD" aqui, para não confundir:

1. **Por baixo dos panos (sempre verdade, com ou sem Spark Connect)**: o Catalyst 
   Optimizer compila o plano lógico do DataFrame num plano físico, que roda sobre um 
   `RDD[InternalRow]` interno (formato binário do Tungsten, gerado via whole-stage 
   codegen). O DAGScheduler que distribui as Tasks nos workers é o mesmo scheduler 
   baseado em RDD de sempre — DataFrame é uma camada de otimização em cima do RDD, 
   não uma substituição dele.
2. **Na API exposta ao cliente (isso sim é específico do Spark Connect)**: no Spark 
   clássico dava pra "descer" a esse RDD chamando `df.rdd`. O cliente Python do Spark 
   Connect **remove essa API de propósito** — ele só fala plano lógico não resolvido via 
   gRPC, então nem existe como chamar `df.rdd` a partir do seu processo local.

Para inspecionar as partitions a partir do cliente Spark Connect, use 
`spark_partition_id()` como uma coluna comum.

📌 Cada linha pertencerá a uma partition diferente. A contagem por `partition_id` revela 
como os dados estão distribuídos entre os workers.

In [ ]:
from pyspark.sql.functions import spark_partition_id

# Agrupa por partition_id para ver quantas linhas cada partição contém
# spark_partition_id() é uma função que retorna o ID da partição de cada linha
partition_counts = (
    sdf_vendas.groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id")   # Ordena para facilitar a leitura
)
partition_counts.show(20)
print(f"Total partitions: {partition_counts.count()}")

📌 **Análise da distribuição**:

Observe se as partições têm tamanhos equilibrados (ideal) ou se há skew (algumas 
partições com muito mais linhas que outras). Dados desbalanceados podem causar 
Tasks lentas que atrasam o Job inteiro (problema de *data skew*).

💡 **Dica**: se houver skew, considere reparticionar com `.repartition(n, col)` ou 
habilitar AQE (Adaptive Query Execution) para balanceamento automático.

In [ ]:
sdf = (
    sdf_vendas
    .repartition(8)
    .groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id")
)
sdf.show()

## Perguntas de negócio processadas no cluster remoto

🎯 **Objetivo:** ir além de `groupBy`/`agg` (notebook 03) usando **funções de Window** e 
**percentil aproximado** — tudo isso continua rodando nos workers do cluster Docker, 
com seu processo Python apenas montando o plano lógico e recebendo o resultado via gRPC.

📌 Nada muda no *como* usar essas funções por estarmos em Spark Connect — a API de 
DataFrame é idêntica à do modo local. O que muda é só o *onde*: cada `Window`, `groupBy` 
e `orderBy` abaixo dispara um shuffle **entre os 2 workers**, não entre threads locais.

#### 💡 **Exemplo 1:** Top 3 maiores vendas de cada mês (ranking com `Window`)

`groupBy` **colapsa** várias linhas em uma só por grupo — ótimo para totais, ruim quando 
você precisa manter as linhas individuais e apenas **rankeá-las** dentro do grupo. É para 
isso que existe `Window`: define uma "janela" (aqui, um `mes`) e calcula uma função sobre 
as linhas dessa janela, **sem** reduzir a contagem de linhas.

🧠 **Por quê `row_number()` e não `rank()`/`dense_rank()`?** As três numeram a partição 
ordenada, mas se divergem em empates: `row_number()` sempre dá uma posição única 
(desempate arbitrário), `rank()` repete a posição em empates e pula números depois, 
`dense_rank()` repete a posição mas não pula. Para "top 3 de cada mês" queremos 
exatamente 3 linhas por grupo — `row_number()` garante isso mesmo com valores empatados.

In [ ]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Janela: particiona por mes e ordena por valor decrescente DENTRO de cada mes
janela_por_mes = Window.partitionBy("mes").orderBy(col("valor").desc())

top3_por_mes = (
    sdf_vendas
    # row_number() numera as linhas de cada partição — não reduz a contagem de linhas
    .withColumn("posicao", row_number().over(janela_por_mes))
    .filter(col("posicao") <= 3)
    .select("mes", "posicao", "id_venda", "id_funcionario", "valor")
    .orderBy("mes", "posicao")
)
top3_por_mes.show(36)

📌 **Entendendo a saída:**

- Cada `mes` aparece exatamente 3 vezes — uma por posição do ranking (1ª, 2ª e 3ª maior venda).
- Diferente de um `groupBy`, nenhuma linha original foi perdida ou agregada: `Window` só *anota* cada linha com sua posição relativa ao grupo.
- Repare que `id_funcionario` e `id_venda` continuam disponíveis — informação que um `groupBy("mes").agg(max("valor"))` jogaria fora.

⚠️ **Atenção:** `Window` sem `partitionBy` (ex: `Window.orderBy("valor")`) manda **todas** as linhas para uma única partição no cálculo — um shuffle massivo e um gargalo real em datasets grandes. Sempre particione por alguma coluna quando possível.

#### 💡 **Exemplo 2:** Evolução acumulada de vendas dentro de cada ano

Pergunta de negócio: **"Em que ponto do ano estamos em relação à meta acumulada?"** 
Precisamos do total de cada mês **e** da soma corrida (running total) até aquele mês, 
dentro do mesmo ano.

🧠 **Por quê `rowsBetween`?** Por padrão, uma `Window` com `orderBy` já usa um frame de 
`unboundedPreceding` até a linha atual (`currentRow`) — mas deixamos explícito aqui para 
ficar claro: a soma acumulada em cada linha considera **todas as linhas anteriores da 
mesma partição** (mesmo `ano`) até a linha atual, e nenhuma linha posterior.

In [ ]:
from pyspark.sql.functions import sum as spark_sum

# Primeiro reduzimos vendas a um total por ano/mes (mesmo padrão do notebook 03)
vendas_por_periodo = (
    sdf_vendas.groupBy("ano", "mes")
    .agg(spark_sum("valor").alias("total_mes"))
)

# Janela: particiona por ano, ordena por mes, e acumula do início do ano até a linha atual
janela_acumulada = (
    Window.partitionBy("ano")
    .orderBy("mes")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

acumulado_anual = (
    vendas_por_periodo
    .withColumn("total_acumulado", spark_sum("total_mes").over(janela_acumulada))
    .orderBy("ano", "mes")
)
acumulado_anual.show(36)

📌 **Entendendo a saída:**

- `total_acumulado` de dezembro de cada ano deve bater com a soma de todos os `total_mes` daquele ano — é o total anual completo.
- O acumulado **reinicia** a cada novo `ano`, porque a `Window` particiona por `ano` — janeiro de 2025 não soma nada de dezembro de 2024.
- Essa é a mesma lógica de um gráfico de "receita acumulada no ano" (YTD) que você vê em dashboards financeiros.

💡 **Dica:** Trocar `rowsBetween(Window.unboundedPreceding, Window.currentRow)` por `rowsBetween(-2, Window.currentRow)` daria uma **média/soma móvel** dos últimos 3 meses em vez do acumulado do ano inteiro.

#### 💡 **Exemplo 3:** Crescimento percentual mês a mês (`lag`)

Pergunta de negócio: **"O negócio cresceu ou caiu em relação ao mês anterior?"** 
`lag()` "olha para trás" dentro da janela e traz o valor de **N linhas atrás** (padrão: 1 linha). 
Combinando com o `total_mes` atual, calculamos a variação percentual.

⚠️ **Atenção:** o primeiro mês de cada ano não tem "mês anterior" **dentro da mesma partição** 
— `lag()` retorna `null` nesse caso, e qualquer conta em cima de `null` também vira `null`. 
Isso é o comportamento correto: não faz sentido comparar janeiro com dezembro do ano anterior 
aqui, já que a janela está particionada por `ano`.

In [ ]:
from pyspark.sql.functions import round as spark_round, lag

# Mesma partição do Exemplo 2, mas sem frame customizado — só precisamos da linha anterior
janela_mes_anterior = Window.partitionBy("ano").orderBy("mes")

crescimento_mensal = (
    vendas_por_periodo
    .withColumn("total_mes_anterior", lag("total_mes").over(janela_mes_anterior))
    .withColumn(
        "crescimento_pct",
        spark_round(
            (col("total_mes") - col("total_mes_anterior")) / col("total_mes_anterior") * 100, 2
        ),
    )
    .orderBy("ano", "mes")
)
crescimento_mensal.show(36)

📌 **Entendendo a saída:**

- O primeiro mês de cada `ano` mostra `total_mes_anterior` e `crescimento_pct` como `null` — exatamente o comportamento esperado de `lag()` na borda da partição.
- `crescimento_pct` positivo indica um mês melhor que o anterior; negativo, um mês pior.
- Compare com `acumulado_anual` do Exemplo 2: um `crescimento_pct` negativo pontual não significa que o acumulado do ano parou de crescer — só que aquele mês específico vendeu menos que o anterior.

💡 **Dica:** `lead()` faz o mesmo que `lag()`, mas "olhando para frente" — útil para comparar cada linha com a **próxima**, em vez da anterior.

#### 💡 **Exemplo 4:** Limiar das vendas "grandes" com `percentile_approx`

Pergunta de negócio: **"A partir de que valor uma venda entra no top 10% (contas VIP)?"**

🧠 **Por quê "approx"?** Um percentil **exato** exige ordenar o dataset inteiro (como fizemos 
com `orderBy` no notebook 02/03) e depois contar posições — caro em qualquer volume grande, 
e pior ainda num cluster: exige um shuffle completo para reunir os dados em ordem total 
entre os workers. `percentile_approx()` usa um algoritmo aproximado (baseado em quantile 
sketches) que estima o percentil sem precisar dessa ordenação global, trocando uma margem 
de erro configurável (parâmetro `accuracy`, padrão 10.000) por uma execução muito mais leve.

In [ ]:
from pyspark.sql.functions import percentile_approx

# 0.9 = P90 (top 10%), 0.99 = P99 (top 1%) — ambos numa única passada sobre os dados
limiares = sdf_vendas.select(
    percentile_approx("valor", 0.9).alias("p90_valor"),
    percentile_approx("valor", 0.99).alias("p99_valor"),
)
limiares.show()

# Quantas vendas realmente ultrapassam o P90 estimado? Confere se a aproximação é razoável
p90 = limiares.first()["p90_valor"]
total = sdf_vendas.count()
acima_p90 = sdf_vendas.filter(col("valor") > p90).count()
print(f"Vendas acima do P90 estimado (R$ {p90:,.2f}): {acima_p90:,} de {total:,} ({acima_p90/total:.1%})")

📌 **Entendendo a saída:**

- `p90_valor` é o valor a partir do qual uma venda está entre as 10% maiores — o limiar prático para definir "conta VIP".
- A conferência final mostra que a fração real de vendas acima do `p90` estimado fica **bem próxima** de 10% — a aproximação é boa o suficiente para decisão de negócio.
- Repare que não precisamos de `orderBy` nem `limit` aqui — diferente do "10 maiores vendas" do notebook 03, a pergunta agora é sobre um **limiar de distribuição**, não sobre registros individuais.

⚠️ **Atenção:** `percentile_approx` é uma aproximação — não use quando o requisito de negócio exigir o percentil **exato** (ex: cálculos regulatórios/contábeis). Para isso, a alternativa exata é `percentile()`/`approx_percentile(..., accuracy=1.0)` ou um `orderBy` completo, ambos mais caros computacionalmente.

---
🎉 **Parabéns!** Você completou o notebook 06 de PySpark.

Você aprendeu:
- Conectar a um cluster Spark remoto via **Spark Connect** (`sc://`) e confirmar que a computação roda nos workers, não no seu laptop
- Que o cliente Spark Connect não expõe `df.rdd` de propósito, embora o DataFrame continue sendo executado sobre RDDs internos (`RDD[InternalRow]`) por baixo dos panos
- Inspecionar e reparticionar dados com `spark_partition_id()` e `repartition()`
- **Funções de Window** (`row_number`, `lag`) para ranking dentro de grupo e comparação com a linha anterior — sem colapsar linhas como o `groupBy` faz
- Soma acumulada com `rowsBetween` para métricas do tipo "acumulado no ano" (YTD)
- `percentile_approx` para estimar limiares de distribuição sem pagar o custo de uma ordenação global no cluster

▶️ **Próximo:** Notebook 07 — Shuffle e Broadcast Join
